# מעבדה 09 — אנרגיית הגל

מודול 08 הסתיים בשאלה שלא יכול היה לענות עליה. פולס חוצה מיתר ומכה בקצה הרחוק; המיתר עצמו אינו
הולך לשום מקום. משהו חצה, והוא לא היה מיתר. כאן תגלו מה הוא היה, וכמה מהר הוא נע.

תפרידו בין שלוש מהירויות שקל לערבב ביניהן, תגלו שהאנרגיה של גל מתקדם יושבת במקום שבו ההזזה שלו
אינה, תראו את שתי צפיפויות האנרגיה עוקבות זו אחר זו נקודה אחר נקודה, תבדקו את חוק השימור המקומי
הקושר אותן לשטף, תמדדו את ההספק החוצה נקודה, ותשוו בין גל מתקדם לגל עומד כמספקי אנרגיה.

החלק האחרון הוא מדידה עם סרגל שגיאה, והוא נושא אזהרה לגבי מדידת כל דבר ההולך כריבוע של אות —
שמכאן ועד סוף הקורס זה כמעט הכול.

עבדו לפי הסדר. במקום שבו המחברת מבקשת שתנבאו, כתבו את הניבוי שלכם בתא המיועד **לפני** הרצת התא
הבא.

## מפרט המודל

| | |
|---|---|
| **מערכת** | המיתר של מודול 08 — הזזה לרוחב $y(x,t)$, מתיחות $T$, צפיפות אורכית $\mu$ — הנושא את השדות הנגזרים $u_K$, $u_P$ ו-$P$ |
| **דינמיקה** | $\mu\,y_{tt} = T\,y_{xx}$; גדלי האנרגיה הם אבחונים המחושבים מן הפתרון, לעולם אינם מוזנים חזרה אליו |
| **שפה** | קצוות קבועים; שום אנרגיה אינה חוצה קצה כזה, שכן קצה קבוע אינו יכול לנוע |
| **אנסמבל** | דטרמיניסטי; רעש גלאים, במקום שבו הוא מתווסף, הוא גאוסי, בלתי תלוי ומוזרע |
| **מוזנח** | בידוד אנרגיה, תנועה אורכית, והתנע שגל רוחבי אולי נושא ואולי לא |
| **תקף כאשר** | השיפועים קטנים, $\lvert y_x \rvert \ll 1$ — $u_P$ זקוקה לכך סדר אחד עמוק יותר ממשוואת התנועה |
| **אופני כישלון** | קריאת הצפיפויות כמדויקות עבור גלים תלולים; הנחה שאנרגיה של מיתר מאבד נשמרת; אמון ב"מהירות האנרגיה" ברגע שהתווך נעשה נפיץ |

כל הפיזיקה נמצאת ב-`wavelab.waves` — פתחו וקראו.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import measurement, waves
from wavelab.validation import scaling_exponent, seed_study

TENSION = 4.0  # N
MU = 0.01  # kg/m, mass per unit length
SPEED = waves.wave_speed(TENSION, MU)  # sqrt(T/mu)
MM = 1e3  # metres to millimetres, for plotting only
MJ = 1e3  # joules per metre to millijoules per metre, and watts to milliwatts

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2026)

print(f"v = sqrt(T/mu) = {SPEED} m/s")
print(f"the string's impedance sqrt(T mu) = {np.sqrt(TENSION * MU)} kg/s")

## חלק 1 — שלוש מהירויות, על פולס אחד

לגל מתקדם יש שלוש מהירויות, והן אינן אותו גודל חובש שם אחד.

- ה**מהירות לרוחב** $\partial y/\partial t$: כמה מהר נע חלק של מיתר. היא הצידה, היא מתהפכת,
  ומי שיצר את הגל בחר בה.
- **מהירות התבנית** $v = \sqrt{T/\mu}$: כמה מהר נעה הצורה. התווך בחר בה.
- **מהירות הובלת האנרגיה** $P/u$: כמה מהר נעה האנרגיה, כאשר $P$ הוא ההספק החוצה נקודה ו-$u$
  האנרגיה ליחידת אורך שם.

הריצו את התא וקראו את כל השלוש מפולס גאוסי אחד.

In [ ]:
LENGTH, CELLS = 4.0, 1600
dx = LENGTH / CELLS
x = np.arange(CELLS + 1) * dx
WIDTH, START, HEIGHT = 0.12, 0.8, 0.008


def gaussian(s):
    """A hump 8 mm high and 12 cm wide, centred at 0.8 m."""
    return HEIGHT * np.exp(-(((s - START) / WIDTH) ** 2))


# A pure right-mover starts with y_t = -v y_x. Give it any other velocity and some of it goes
# the other way, which is what module 08's pluck did with y_t = 0.
pulse_slope = -2.0 * (x - START) / WIDTH**2 * gaussian(x)
dt = 0.5 * dx / SPEED
n_steps = int(round(2.0 / SPEED / dt))
pulse = waves.simulate_string(gaussian(x), -SPEED * pulse_slope, dx, dt, TENSION, MU, n_steps,
                              save_every=max(1, n_steps // 200))

slope = np.gradient(pulse.y, dx, axis=1)
kinetic = waves.kinetic_density(pulse.dydt, MU)
potential = waves.potential_density(slope, TENSION)
flux = waves.energy_flux(slope, pulse.dydt, TENSION)
density = kinetic + potential

# P/u is 0/0 wherever the string is flat and still, so it is read only where there is a wave.
carrying = density > 1e-3 * density.max()
energy_speed = flux[carrying] / density[carrying]

fig, (shape_axis, speed_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
snapshot = pulse.times.size // 2
shape_axis.plot(x, MM * pulse.y[snapshot], color="0.15", label="y")
shape_axis.set_xlabel("x (m)"), shape_axis.set_ylabel("y (mm)"), shape_axis.legend(fontsize=8)
speed_axis.plot(x, pulse.dydt[snapshot], color="tab:orange", label="dy/dt (m/s)")
speed_axis.axhline(0, color="0.85", lw=0.8)
speed_axis.set_xlabel("x (m)"), speed_axis.set_ylabel("transverse velocity (m/s)")
speed_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"fastest any piece of string moves : {np.abs(pulse.dydt).max():.3f} m/s")
print(f"the pattern moves at              : {SPEED} m/s")
print(f"the energy moves at               : {energy_speed.min():.6f} to "
      f"{energy_speed.max():.6f} m/s")
print()
print(f"The string's own motion is {np.abs(pulse.dydt).max() / SPEED:.4f} of the wave speed, and")
print("it is sideways. The energy's motion is the wave speed, to seven figures, and it is")
print("along the string. Two of the three are the same number for a reason; one is a free knob.")

## חלק 2 — לנבא

התחייבו לתשובה לכל אחת לפני שתריצו משהו למטה.

1. גל סינוסי נע לאורך מיתר. האם האנרגיה ליחידת אורך גדולה ביותר בפסגות, בחציות האפס, או שהיא
   זהה בכל מקום?
2. האם חלק של מיתר יכול אי פעם לנוע מהר יותר מן הגל הנע לאורכו? אם כן, איך היה נראה הגל הזה?
3. אתם מכפילים את משרעת הגל הסינוסי ומשאירים את כל השאר קבוע. ההספק שהוא מספק עולה בגורם של
   כמה?
4. גל מתקדם וגל עומד באותה משרעת ובאותה תדירות עוברים כל אחד נקודה באמצע המיתר. איזה מהם מספק
   יותר אנרגיה מעבר לנקודה הזו במשך שנייה אחת?

**הניבוי שלכם:**

## חלק 3 — היכן יושבת האנרגיה

זהו הניסוי המפריך של המודול, והוא דורש שרטוט אחד.

המיתר אינו יכול לאגור אנרגיה בכך שהוא *מוזז* — הרימו קטע ישר בשלמותו ולא מתחתם דבר ולא הנעתם
דבר. הוא אוגר אנרגיה בכך שהוא *מתוח*, מה שדורש שיפוע, ובכך שהוא *נע*, מה שדורש מהירות לרוחב.
לפסגה אין אף אחת מהן: המיתר שם שטוח, ולאותו רגע, במנוחה.

התא משגר רכבת סינוסית ומשרטט את צפיפות האנרגיה ישירות מתחת למיתר. השוו בין שני הפאנלים עמודה
אחר עמודה.

In [ ]:
TRAIN_LENGTH, TRAIN_CELLS = 8.0, 1280
tdx = TRAIN_LENGTH / TRAIN_CELLS
tx = np.arange(TRAIN_CELLS + 1) * tdx
AMPLITUDE = 0.002  # 2 mm
FLAT = (0.4, 3.2)  # the window is exactly 1 between these, with 0.4 m shoulders
EDGE = 0.4
tdt = 0.5 * tdx / SPEED
TRAIN_STEPS = int(round(0.20 / tdt))
GATE = int(round(4.2 / tdx))


def train(s, amplitude, wavelength):
    """A sine inside a flat-topped window, returned with its exact slope.

    A travelling sine cannot live on a string with fixed ends — it would have to move them —
    so the train is windowed and kept clear of both walls. The slope is written down rather
    than differenced, because launching a pure right-mover needs y_t = -v y_x exactly: get the
    slope 1% wrong and 1% of the wave sets off the other way.
    """
    k = 2 * np.pi / wavelength
    left = np.clip((s - (FLAT[0] - EDGE)) / EDGE, 0.0, 1.0)
    right = np.clip(((FLAT[1] + EDGE) - s) / EDGE, 0.0, 1.0)
    ramp = left * right
    envelope = 0.5 * (1.0 - np.cos(np.pi * ramp))
    d_ramp = (np.where((left > 0) & (left < 1), 1.0 / EDGE, 0.0) * right
              + left * np.where((right > 0) & (right < 1), -1.0 / EDGE, 0.0))
    d_envelope = 0.5 * np.pi * np.sin(np.pi * ramp) * d_ramp
    return (amplitude * np.sin(k * s) * envelope,
            amplitude * (k * np.cos(k * s) * envelope + np.sin(k * s) * d_envelope))


def launch(amplitude, wavelength, n_steps=TRAIN_STEPS):
    y0, dydx0 = train(tx, amplitude, wavelength)
    return waves.simulate_string(y0, -SPEED * dydx0, tdx, tdt, TENSION, MU, n_steps)


WAVELENGTH = 0.4
OMEGA = SPEED * 2 * np.pi / WAVELENGTH
train_run = launch(AMPLITUDE, WAVELENGTH)

look = TRAIN_STEPS // 3
train_y = train_run.y[look]
train_slope = np.gradient(train_y, tdx)
train_u = (waves.kinetic_density(train_run.dydt[look], MU)
           + waves.potential_density(train_slope, TENSION))
window = (tx > 1.2) & (tx < 2.2)

fig, (wave_axis, energy_axis) = plt.subplots(2, 1, figsize=(10, 4.4), sharex=True)
wave_axis.plot(tx[window], MM * train_y[window], color="0.15")
wave_axis.axhline(0, color="0.9", lw=0.8)
wave_axis.set_ylabel("y (mm)")
energy_axis.fill_between(tx[window], MJ * train_u[window], alpha=0.3)
energy_axis.plot(tx[window], MJ * train_u[window], lw=1.3)
energy_axis.set_xlabel("x (m)"), energy_axis.set_ylabel("u (mJ/m)")
plt.tight_layout()
plt.show()

crest = np.argmax(np.abs(train_y[window]))
richest = np.argmax(train_u[window])
print(f"tallest point in the window : x = {tx[window][crest]:.3f} m, "
      f"|y| = {MM * abs(train_y[window][crest]):.3f} mm, "
      f"u = {MJ * train_u[window][crest]:.5f} mJ/m")
print(f"most energetic point        : x = {tx[window][richest]:.3f} m, "
      f"|y| = {MM * abs(train_y[window][richest]):.4f} mm, "
      f"u = {MJ * train_u[window][richest]:.5f} mJ/m")
print()
print(f"The crest holds {train_u[window][crest] / train_u[window].max():.1e} of the energy the")
print("zero crossing holds. The energy peaks where the displacement vanishes, and the energy")
print("humps come twice as often as the crests do, because u goes as the square of a sine.")

## חלק 4 — שתי צפיפויות, שטף אחד, והחוק הקושר ביניהם

עכשיו המאזן, על פולס ולא על רכבת.

$$
u_K = \tfrac12 \mu \left(\frac{\partial y}{\partial t}\right)^2,
\qquad
u_P = \tfrac12 T \left(\frac{\partial y}{\partial x}\right)^2,
\qquad
P = -T\,\frac{\partial y}{\partial x}\,\frac{\partial y}{\partial t}.
$$

שלושה דברים לחפש, לפי סדר ההפתעה שבהם.

1. $u_K$ ו-$u_P$ מונחות זו על זו — לא בממוצע, בכל נקודה.
2. $P$ חלקי $v$ נוחת על $u_K + u_P$: האנרגיה נעה במהירות הגל.
3. $\partial u/\partial t + \partial P/\partial x = 0$, חוק השימור המקומי הראשון בקורס. שום דבר
   בפותר אינו יודע עליו.

In [ ]:
fig, (shape_axis, density_axis) = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
shape_axis.plot(x, MM * pulse.y[snapshot], color="0.15")
shape_axis.axhline(0, color="0.9", lw=0.8)
shape_axis.set_ylabel("y (mm)")
density_axis.fill_between(x, MJ * density[snapshot], color="0.6", alpha=0.25,
                          label="u_K + u_P")
density_axis.plot(x, MJ * potential[snapshot], lw=5, alpha=0.45, color="tab:orange", label="u_P")
density_axis.plot(x, MJ * kinetic[snapshot], lw=1.4, color="tab:blue", label="u_K")
density_axis.plot(x, MJ * flux[snapshot] / SPEED, lw=1.5, ls=":", color="tab:green",
                  label="P / v")
density_axis.set_xlabel("x (m)"), density_axis.set_ylabel("u (mJ/m)")
density_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

peak = density[snapshot].max()
print(f"largest |u_K - u_P| anywhere : "
      f"{np.max(np.abs(kinetic - potential)) / peak:.1e} of the peak")
print(f"largest |P/v - u| anywhere   : "
      f"{np.max(np.abs(flux / SPEED - density)) / peak:.1e} of the peak")

# The conservation law needs neighbouring steps in time, so this short run keeps every one.
check = waves.simulate_string(gaussian(x), -SPEED * pulse_slope, dx, dt, TENSION, MU, 200)
middle = 100


def total_density(step):
    return (waves.kinetic_density(check.dydt[step], MU)
            + waves.potential_density(np.gradient(check.y[step], dx), TENSION))


rate = (total_density(middle + 1) - total_density(middle - 1)) / (2 * dt)
divergence = np.gradient(
    waves.energy_flux(np.gradient(check.y[middle], dx), check.dydt[middle], TENSION), dx
)
residual = np.max(np.abs(rate + divergence)) / np.max(np.abs(divergence))
print(f"du/dt + dP/dx, as a fraction of dP/dx : {residual:.1e}")
print()
print("A pulse eight millimetres tall carries a double-humped energy, because both densities go")
print("as the square of the slope and a hump is flat on top. The lesson of part 3 again: it is")
print("the steep parts of a wave that are loaded, not the tall ones.")

## חלק 5 — מד ההספק

שטף הוא הבטחה: אינטגרלו את $P$ על הזמן בנקודה קבועה ותקבלו את האנרגיה שהגיעה בסופו של דבר
מעברה. המחצית הראשונה של התא בודקת את ההבטחה הזו על הפולס. המחצית השנייה מפנה את אותו מכשיר אל
הרכבת הסינוסית ומודדת את ההספק הממוצע, שהתאוריה אומרת שהוא

$$
\langle P \rangle = \tfrac12\,\mu\,v\,\omega^2 A^2 .
$$

שימו לב לשני המעריכים. יש סיבה אחת לשני הריבועים, וכדאי למצוא אותה.

In [ ]:
CROSSING = int(round(2.0 / dx))
delivered_run = waves.simulate_string(gaussian(x), -SPEED * pulse_slope, dx, dt, TENSION, MU,
                                      int(round(2.4 / SPEED / dt)))
gate_slope = np.gradient(delivered_run.y, dx, axis=1)[:, CROSSING]
gate_flux = waves.energy_flux(gate_slope, delivered_run.dydt[:, CROSSING], TENSION)
delivered = np.trapezoid(gate_flux, delivered_run.times)
beyond = waves.total_energy(delivered_run.y[-1][CROSSING:], delivered_run.dydt[-1][CROSSING:],
                            dx, TENSION, MU)
started = waves.total_energy(delivered_run.y[0], delivered_run.dydt[0], dx, TENSION, MU)
print(f"the pulse set out with              : {1e3 * started:.6f} mJ")
print(f"the wattmeter at 2 m counted        : {1e3 * delivered:.6f} mJ")
print(f"weighing the string beyond 2 m gives: {1e3 * beyond:.6f} mJ")
print(f"wattmeter / weighing                : {delivered / beyond:.6f}")
print()

WINDOW = (0.055, 0.185)  # while the train's flat top is passing the gate at 4.2 m


def mean_power(amplitude, wavelength):
    """Average the flux at the gate over whole cycles of the train's flat top."""
    run = launch(amplitude, wavelength)
    omega = SPEED * 2 * np.pi / wavelength
    slope_at_gate = np.gradient(run.y, tdx, axis=1)[:, GATE]
    watts = waves.energy_flux(slope_at_gate, run.dydt[:, GATE], TENSION)
    first = int(np.searchsorted(run.times, WINDOW[0]))
    cycles = int((WINDOW[1] - WINDOW[0]) / (2 * np.pi / omega))
    last = int(np.searchsorted(run.times, run.times[first] + cycles * 2 * np.pi / omega))
    return float(watts[first:last].mean()), omega, watts


measured, _, gate_watts = mean_power(AMPLITUDE, WAVELENGTH)
fig, (trace_axis, fit_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
trace_axis.plot(1e3 * train_run.times, MJ * gate_watts, lw=1.0)
trace_axis.axhline(MJ * measured, color="crimson", lw=1.0, ls="--", label="mean over whole cycles")
trace_axis.axhline(0, color="0.85", lw=0.8)
trace_axis.set_xlabel("t (ms)"), trace_axis.set_ylabel("P at the gate (mW)")
trace_axis.legend(fontsize=8)

amplitudes = np.array([0.001, 0.002, 0.003])
by_amplitude = np.array([mean_power(a, WAVELENGTH)[0] for a in amplitudes])
wavelengths = np.array([0.8, 0.4, 0.2])
swept = [mean_power(AMPLITUDE, lam) for lam in wavelengths]
frequencies = np.array([row[1] for row in swept])
by_frequency = np.array([row[0] for row in swept])
fit_axis.loglog(MM * amplitudes, MJ * by_amplitude, "o", label="sweep A at 50 Hz")
fit_axis.loglog(MM * amplitudes, MJ * np.array(
    [waves.sinusoidal_mean_power(a, OMEGA, TENSION, MU) for a in amplitudes]
), lw=0.9, ls="--", color="0.5", label="(1/2) mu v omega^2 A^2")
fit_axis.set_xlabel("amplitude (mm)"), fit_axis.set_ylabel("mean power (mW)")
fit_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"measured <P> at A = 2 mm, 50 Hz : {MJ * measured:.4f} mW")
print(f"(1/2) mu v omega^2 A^2          : "
      f"{MJ * waves.sinusoidal_mean_power(AMPLITUDE, OMEGA, TENSION, MU):.4f} mW")
print(f"fitted exponent in amplitude    : {scaling_exponent(amplitudes, by_amplitude):.6f}")
print(f"fitted exponent in frequency    : {scaling_exponent(frequencies, by_frequency):.6f}")
print()
print("The frequency exponent falls a little short of 2, and the shortfall is the shortest")
print("wave's. At 100 Hz the wavelength spans 32 grid points, and a centred difference reads")
print("the slope of a sine low by (k dx)^2/6, which is 0.6% — the measurement's error, not the")
print("string's. The amplitude exponent has no such excuse and comes out exact.")

## חלק 6 — מתקדם מול עומד

גל עומד אלים כמו גל מתקדם: אותה משרעת, אותה תדירות, אותה אנרגיה מתנדנדת. בקשו ממנו לספק אנרגיה
מעבר לנקודה, ולא יספק דבר.

התא מציב אופן של מיתר קבוע לצד הרכבת מחלק 5 ומודד את שניהם.

In [ ]:
STANDING_LENGTH, STANDING_CELLS = 1.0, 500
sdx = STANDING_LENGTH / STANDING_CELLS
sx = np.arange(STANDING_CELLS + 1) * sdx
MODE = 4  # four half wavelengths between the walls, so lambda = 0.5 m
sk = MODE * np.pi / STANDING_LENGTH
somega = SPEED * sk
sdt = 0.5 * sdx / SPEED
per_period = int(round((2 * np.pi / somega) / sdt))
standing = waves.simulate_string(AMPLITUDE * np.sin(sk * sx), np.zeros_like(sx), sdx, sdt,
                                 TENSION, MU, 4 * per_period)

standing_slope = np.gradient(standing.y, sdx, axis=1)
standing_flux = waves.energy_flux(standing_slope, standing.dydt, TENSION)
standing_kinetic = waves.kinetic_density(standing.dydt, MU)
standing_potential = waves.potential_density(standing_slope, TENSION)
SGATE = int(round(0.3125 / sdx))  # an eighth of a wavelength from a node, where P is largest
averaged = standing_flux[: 4 * per_period].mean(axis=0)

fig, (flux_axis, budget_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
flux_axis.plot(1e3 * standing.times, MJ * standing_flux[:, SGATE], lw=1.0, color="tab:orange")
flux_axis.axhline(0, color="0.4", lw=1.0)
flux_axis.set_xlabel("t (ms)"), flux_axis.set_ylabel("P at the gate (mW)")
budget_axis.plot(1e3 * standing.times,
                 MJ * np.trapezoid(standing_kinetic, dx=sdx, axis=1), label="kinetic")
budget_axis.plot(1e3 * standing.times,
                 MJ * np.trapezoid(standing_potential, dx=sdx, axis=1), label="potential")
budget_axis.plot(1e3 * standing.times,
                 MJ * waves.total_energy(standing.y, standing.dydt, sdx, TENSION, MU),
                 color="0.3", label="total")
budget_axis.set_xlabel("t (ms)"), budget_axis.set_ylabel("energy (mJ)")
budget_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

energies = np.asarray(waves.total_energy(standing.y, standing.dydt, sdx, TENSION, MU))
mean_kinetic = np.trapezoid(standing_kinetic, dx=sdx, axis=1)[: 4 * per_period].mean()
mean_potential = np.trapezoid(standing_potential, dx=sdx, axis=1)[: 4 * per_period].mean()
print(f"peak instantaneous |P| at the gate     : {MJ * np.abs(standing_flux).max():.3f} mW")
print(f"|time-averaged P|, largest anywhere    : "
      f"{MJ * np.abs(averaged).max():.3e} mW")
print(f"ratio                                  : "
      f"{np.abs(averaged).max() / np.abs(standing_flux).max():.1e}")
print(f"total energy varies over the run by    : {np.ptp(energies) / energies[0]:.1e} of itself")
print(f"cycle-averaged kinetic / potential     : {mean_kinetic / mean_potential:.5f}")
pointwise_gap = (np.max(np.abs(standing_kinetic - standing_potential))
                 / np.max(standing_kinetic + standing_potential))
print(f"largest pointwise |u_K - u_P| / max u  : {pointwise_gap:.4f}")
print()
print("Two contrasts with the travelling wave, in the last two numbers. The standing wave's")
print("kinetic and potential energies are equal only after averaging over a cycle — pointwise")
print("they are as unequal as two quantities can be, since at the moment of release the string")
print("is all potential and a quarter period later all kinetic. And its flux, large at every")
print("instant, averages to nothing at every point: a standing wave lends energy and takes it")
print("back twice a cycle, and transports none.")

## חלק 7 — מדידה, עם סרגל שגיאה

מד הספק אמיתי אינו יכול לקרוא את $y_x$ ואת $y_t$ מתוך פותר. נניח במקום זאת שאתם מצלמים את
המיתר ומשחזרים את ההזזה של שלוש נקודות שכנות, עם רעש של 5% ממשרעת הגל בכל דגימה. אז יש לגזור
את שתי הנגזרות מתוך התמונה הזו.

יש שתי דרכים הגונות לעשות זאת, והן אינן מסכימות.

- **הפרשים מוצלבים:** קחו את השיפוע משני השכנים במרחב ואת המהירות משני השכנים בזמן, ואז בנו
  $P = -T\,y_x\,y_t$.
- **שיפוע בריבוע:** עבור גל הנע ימינה $y_t = -v\,y_x$, ולכן $P = T v\,y_x^2$, והשיפוע לבדו
  מספיק. פחות מספרים, אותה אלגברה.

נבאו במי מהם הייתם בוטחים, ואז הריצו את התא.

In [ ]:
patch = train_run.y[:, GATE - 1 : GATE + 2]  # the three columns a camera would see
NOISE = 0.05 * AMPLITUDE
first = int(np.searchsorted(train_run.times, WINDOW[0]))
cycles = int((WINDOW[1] - WINDOW[0]) / (2 * np.pi / OMEGA))
last = int(np.searchsorted(train_run.times, train_run.times[first] + cycles * 2 * np.pi / OMEGA))
averaging = slice(first, last)
EXACT = waves.sinusoidal_mean_power(AMPLITUDE, OMEGA, TENSION, MU)


def by_cross_differencing(generator):
    noisy = measurement.add_noise(patch, NOISE, generator)
    space = (noisy[:, 2] - noisy[:, 0]) / (2 * tdx)
    time_derivative = (noisy[2:, 1] - noisy[:-2, 1]) / (2 * tdt)
    return float(waves.energy_flux(space[1:-1], time_derivative, TENSION)[averaging].mean())


def by_slope_squared(generator):
    noisy = measurement.add_noise(patch, NOISE, generator)
    space = (noisy[:, 2] - noisy[:, 0]) / (2 * tdx)
    return float((TENSION * SPEED * space**2)[averaging].mean())


cross = seed_study(by_cross_differencing, n_seeds=32, base_seed=9)
squared = seed_study(by_slope_squared, n_seeds=32, base_seed=9)

fig, (record_axis, spread_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
noisy_column = measurement.add_noise(patch[:, 1], NOISE, rng)
record_axis.plot(1e3 * train_run.times, MM * noisy_column, lw=0.7, color="0.5", label="noisy")
record_axis.plot(1e3 * train_run.times, MM * patch[:, 1], lw=1.0, color="k", label="underneath")
record_axis.set_xlim(60, 80)
record_axis.set_xlabel("t (ms)"), record_axis.set_ylabel("y (mm)"), record_axis.legend(fontsize=8)
spread_axis.hist(MJ * cross.values, bins=10, alpha=0.7, label="cross-differenced")
spread_axis.hist(MJ * squared.values, bins=10, alpha=0.7, label="slope-squared")
spread_axis.axvline(MJ * EXACT, color="k", lw=1.2, ls="--", label="(1/2) mu v omega^2 A^2")
spread_axis.set_xlabel("measured mean power (mW)"), spread_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

sigma_slope = NOISE * np.sqrt(2) / (2 * tdx)  # the noise the space difference puts on y_x
predicted_offset = TENSION * SPEED * sigma_slope**2
print(f"truth                : {MJ * EXACT:.3f} mW")
print(f"cross-differenced    : ({MJ * cross.mean:.3f} +/- {MJ * cross.standard_error:.3f}) mW, "
      f"agrees within 3 sigma: {cross.agrees_with(EXACT, 3.0)}")
print(f"slope-squared        : ({MJ * squared.mean:.3f} +/- {MJ * squared.standard_error:.3f}) mW, "
      f"agrees within 3 sigma: {squared.agrees_with(EXACT, 3.0)}")
print()
print(f"excess of the second over the first : {MJ * (squared.mean - cross.mean):.3f} mW")
print(f"T v sigma_slope^2                   : {MJ * predicted_offset:.3f} mW")
print()
print("The second estimator is biased, and no number of seeds will fix it. Squaring a noisy")
print("slope adds the noise's own variance to the answer: <(y_x + d)^2> = <y_x^2> + sigma_d^2,")
print("every time, in the same direction. The first estimator multiplies two noises drawn from")
print("different samples — neighbours in space, neighbours in time — which are independent, so")
print("their cross term averages away. Everything you will measure after this module, from")
print("intensity to irradiance to power, goes as the square of an amplitude. This is the trap.")

In [ ]:
# 1. Three velocities: the string crawls, the pattern and the energy travel at v.
assert np.abs(pulse.dydt).max() < 0.1 * SPEED
assert np.allclose(energy_speed, SPEED, rtol=1e-4)

# 2. The falsifier: the crest is the emptiest place, not the fullest.
assert train_u[window][crest] < 1e-2 * train_u[window].max()
assert abs(train_y[window][richest]) < 1e-2 * abs(train_y[window][crest])

# 3. Equipartition, the energy velocity, and the local conservation law.
assert np.max(np.abs(kinetic - potential)) < 1e-3 * peak
assert np.max(np.abs(flux / SPEED - density)) < 1e-3 * peak
assert residual < 1e-2

# 4. The wattmeter accounts for the pulse, and the mean power is the closed form.
assert abs(delivered / beyond - 1) < 1e-3
assert abs(measured / waves.sinusoidal_mean_power(AMPLITUDE, OMEGA, TENSION, MU) - 1) < 5e-3
assert abs(scaling_exponent(amplitudes, by_amplitude) - 2.0) < 1e-3
assert abs(scaling_exponent(frequencies, by_frequency) - 2.0) < 1e-2

# 5. A standing wave transports nothing, and stores a constant total.
assert np.abs(averaged).max() < 1e-5 * np.abs(standing_flux).max()
assert np.ptp(energies) / energies[0] < 1e-3
assert abs(mean_kinetic / mean_potential - 1) < 1e-3

# 6. The honest wattmeter agrees through the noise; the tempting one does not.
assert cross.agrees_with(EXACT, n_sigma=3.0)
assert not squared.agrees_with(EXACT, n_sigma=3.0)
assert abs((squared.mean - cross.mean) / predicted_offset - 1) < 0.05

print("all checks passed")
print(f"<P> measured through noise: ({MJ * cross.mean:.2f} +/- {MJ * cross.standard_error:.2f}) mW "
      f"against (1/2) mu v omega^2 A^2 = {MJ * EXACT:.2f} mW")

## חלק 8 — חקרו בעצמכם

שגרו רכבת משלכם ומדדו אותה. שלושה דברים שכדאי לחפש.

- חצו את המשרעת והכפילו את התדירות. ההספק הממוצע אמור לחזור למקום שבו היה — והמיתר אמור להיראות
  שונה לגמרי.
- העלו את המתיחות במשרעת ובתדירות קבועות. ההספק עולה, אף שדבר בצורת הגל במרחב לא השתנה. איזה
  גורם ב-$\tfrac12\mu v\omega^2A^2$ עשה זאת?
- העלו את המשרעת עד שהשיפוע מפסיק להיות קטן. המספרים ממשיכים לזרום; המודול הפסיק לחול. התבוננו
  בשיפוע המודפס, לא בשרטוט.

In [ ]:
import ipywidgets as widgets


def explore(tension=4.0, amplitude_mm=2.0, frequency_hz=50.0):
    speed = waves.wave_speed(tension, MU)
    amplitude = amplitude_mm / MM
    wavelength = speed / frequency_hz
    omega = 2 * np.pi * frequency_hz
    dt = 0.5 * tdx / speed
    steps = int(round(0.9 * TRAIN_LENGTH / speed / dt))
    y0, dydx0 = train(tx, amplitude, wavelength)
    run = waves.simulate_string(y0, -speed * dydx0, tdx, dt, tension, MU, steps)

    slope_at_gate = np.gradient(run.y, tdx, axis=1)[:, GATE]
    watts = waves.energy_flux(slope_at_gate, run.dydt[:, GATE], tension)
    arrives = (4.2 - FLAT[1]) / speed
    leaves = (4.2 - FLAT[0]) / speed
    start = int(np.searchsorted(run.times, arrives + 0.05 * (leaves - arrives)))
    cycles = max(1, int(0.9 * (leaves - arrives) * frequency_hz))
    stop = int(np.searchsorted(run.times, run.times[start] + cycles / frequency_hz))

    fig, (wave_axis, watt_axis) = plt.subplots(1, 2, figsize=(11, 3.0))
    wave_axis.plot(tx, MM * run.y[steps // 3], lw=0.9)
    wave_axis.axvline(tx[GATE], color="crimson", lw=0.7, ls=":")
    wave_axis.set_xlabel("x (m)"), wave_axis.set_ylabel("y (mm)")
    watt_axis.plot(1e3 * run.times, MJ * watts, lw=0.8)
    watt_axis.axhline(MJ * watts[start:stop].mean(), color="crimson", lw=1.0, ls="--")
    watt_axis.set_xlabel("t (ms)"), watt_axis.set_ylabel("P at the gate (mW)")
    plt.tight_layout()
    plt.show()

    print(f"measured <P> over {cycles} cycles : {MJ * watts[start:stop].mean():.3f} mW")
    print(f"(1/2) mu v omega^2 A^2           : "
          f"{MJ * waves.sinusoidal_mean_power(amplitude, omega, tension, MU):.3f} mW")
    steepest = np.abs(np.gradient(run.y[steps // 3], tdx)).max()
    print(f"steepest slope on the string     : {steepest:.3f}"
          "   (the model needs this well below 1)")


widgets.interact_manual(
    explore,
    tension=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=4.0, description="T (N)"),
    amplitude_mm=widgets.FloatSlider(min=0.5, max=40.0, step=0.5, value=2.0,
                                     description="A (mm)"),
    frequency_hz=widgets.FloatSlider(min=20.0, max=100.0, step=5.0, value=50.0,
                                     description="f (Hz)"),
);

## בדקו את הבנתכם

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "09-wave-energy.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. בחלק 3 התברר שפסגות הגל הן המקומות הריקים ביותר על המיתר. הסבירו זאת למי שבטוח שהאנרגיה
   נמצאת במקום שבו נמצאת ההזזה, תוך שימוש רק בשני הדברים שמיתר יכול לעשות כדי לאגור אנרגיה.
2. בחלק 1 המיתר נע ב-1.1 מ'/ש' בעוד הגל נע ב-20 מ'/ש', ובחלק 4 גם האנרגיה נעה ב-20 מ'/ש'. אמרו
   מדוע שניים מאותם שלושה מספרים היו חייבים להתאים, ומדוע השלישי היה חופשי להיות כל דבר.
3. בחלק 7 שני מדי ההספק היו אלגברה נכונה המיושמת על אותם נתונים, ואחד מהם טעה ברבע מן התשובה.
   אמרו מה הופך מדידה למוטה ולא רק רועשת, ואיך הייתם מזהים את אותה מלכודת בגלאי אור המודד עוצמה.

**התשובות שלכם:**

1.
2.
3.